## Reading 34-2 - The Power of Ten

By the end of this reading, you will learn:
<ol>
    <li>The ten rules Gerard Holzmann wrote for safety-critical code at NASA's Jet Propulsion Laboratory</li>
    <li>Why each rule exists: what failure it makes impossible, or at least detectable</li>
    <li>Rule 1: why flight software does not use recursion</li>
    <li>Rule 2: why every loop needs a fixed upper bound</li>
    <li>Rule 3: how to build a linked list without calling <code>malloc</code></li>
</ol>

### Ten Rules That Fit on One Page

In Reading 34-1, we saw that MISRA C has 225 guidelines. In 2006, Gerard Holzmann, who led the Laboratory for Reliable Software at JPL, published a much shorter list in <i>IEEE Computer</i>. His reasoning was practical: a rule set that is too long to remember is a rule set that people do not follow. He kept it to ten rules, and he chose each one so that <b>a tool could check it automatically</b>.

The rules became the core of the JPL coding standard used for the Mars Science Laboratory, the Curiosity rover. Holzmann compared them to a seat belt: slightly uncomfortable at first, and then something you stop noticing.

<table>
    <tr><th>Rule</th><th>What it says</th><th>What it prevents</th></tr>
    <tr><td>1</td><td>Simple control flow: no <code>goto</code>, no <code>setjmp</code>/<code>longjmp</code>, no recursion</td><td>Stack overflow; control flow tools cannot follow</td></tr>
    <tr><td>2</td><td>Every loop has a fixed upper bound that a tool can prove</td><td>Runaway loops and hangs</td></tr>
    <tr><td>3</td><td>No dynamic memory allocation after initialization</td><td>Leaks, fragmentation, and running out of memory in flight</td></tr>
    <tr><td>4</td><td>Functions fit on one printed page, about 60 lines</td><td>Code too large to review or test as a unit</td></tr>
    <tr><td>5</td><td>At least two assertions per function, on average, with an explicit recovery action</td><td>Errors that go unnoticed until they spread</td></tr>
    <tr><td>6</td><td>Declare data at the smallest possible scope</td><td>Unintended changes from far-away code</td></tr>
    <tr><td>7</td><td>Check every non-<code>void</code> return value, and validate every parameter</td><td>Silent failures that propagate</td></tr>
    <tr><td>8</td><td>Use the preprocessor only for header includes and simple macros</td><td>Code that tools, and people, cannot read</td></tr>
    <tr><td>9</td><td>At most one level of pointer dereference; no function pointers</td><td>Data flow that tools cannot follow</td></tr>
    <tr><td>10</td><td>Compile with all warnings on, with zero warnings; analyze daily with zero warnings</td><td>Defects that a tool already knew about</td></tr>
</table>

You have been following Rule 10 since Homework 1: <code>-Wall -Wextra -Werror</code> is Rule 10 in action. In this reading, we look closely at the first three rules. Lecture 35 covers the rest.

### Rule 1 - No Recursion - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec34/recursive_delete.c">recursive_delete.c</a>

In Lecture 24 and Homework 8, we deleted linked structures recursively:

    void destructor_recursive(sll_node *curr_ptr)
    {
    	if (curr_ptr != NULL) {
    		destructor_recursive(curr_ptr->next_node);
    		free(curr_ptr);
    	}
    }

It is short, elegant, and correct. So why would NASA forbid it?

Every function call uses a block of memory on the <b>stack</b> to hold its parameters, local variables, and return address. That memory is not released until the function returns. A recursive destructor on a list of <i>n</i> nodes has <i>n</i> calls waiting at the same time, so its stack use grows with the size of the data:

    $ ./recursive_delete 1000 recursive
    Built a list of 1000 nodes
    Deleted the list with the recursive destructor
    $ ./recursive_delete 1000000 iterative
    Built a list of 1000000 nodes
    Deleted the list with the iterative destructor
    $ ./recursive_delete 1000000 recursive
    Built a list of 1000000 nodes
    Segmentation fault

The default stack on <code>student13</code> is 8 MB. A flight computer may have a few kilobytes of stack per task. We can simulate that by limiting the stack to 64 KB:

    $ make small_stack
    ulimit -s 64; ./recursive_delete 1000 recursive
    Built a list of 1000 nodes
    Deleted the list with the recursive destructor
    ulimit -s 64; ./recursive_delete 5000 recursive
    Built a list of 5000 nodes
    Segmentation fault

Five thousand nodes is enough. And the code itself never changed, so a test with a short list would pass every time.

The real problem is that <b>no tool can prove how deep the recursion will go</b>, because it depends on the data. Without recursion, a tool can add up the stack used by each function along the deepest chain of calls and prove that the stack is large enough. The fix is the loop version, which uses the same small amount of stack no matter how long the list is:

    void destructor_iterative(sll_node *curr_ptr)
    {
    	while (curr_ptr != NULL) {
    		sll_node *next = curr_ptr->next_node;
    		free(curr_ptr);
    		curr_ptr = next;
    	}
    }

Notice that we save <code>next</code> <i>before</i> calling <code>free</code>. Reading <code>curr_ptr->next_node</code> after freeing <code>curr_ptr</code> would be a use-after-free.

### Rule 2 - Every Loop Has a Fixed Bound - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec34/bounded_find.c">bounded_find.c</a>

Here is a search we might write without a second thought:

    node_t *find_unbounded(node_t *n, int key)
    {
    	while (n->key != key) {
    		n = n->next;
    	}
    	return n;
    }

When will this loop end? Only when it finds the key. That assumes two things: the key is in the list, and the list is intact. On a spacecraft, neither is guaranteed. A single radiation-induced bit flip can change one <code>next</code> pointer so that the list loops back on itself. The example simulates exactly that: node 9 is corrupted to point back to node 4.

    $ ./bounded_find 30 unbounded
    Found key 30
    $ timeout 2 ./bounded_find 35 unbounded
    List built. Node 9 now points back to node 4.
    $ echo $?
    124

Searching for a key that exists works fine, so the bug passes any test that only looks up existing keys. Searching for a missing key loops forever, until <code>timeout</code> kills the program after 2 seconds. The exit status 124 means "killed by <code>timeout</code>." On a spacecraft, the equivalent of <code>timeout</code> is a hardware <b>watchdog</b> timer that resets the computer. Mars Pathfinder's watchdog resets in 1997 were a real-world example of software that stopped making progress.

The compliant version puts a fixed, compile-time bound on the loop and treats hitting the bound as an anomaly to report:

    for (int i = 0; i < MAX_NODES; i++) {
    	if (n == NULL) {
    		return ERR_NOT_FOUND;
    	}
    	if (n->key == key) {
    		*out = n;
    		return OK;
    	}
    	n = n->next;
    }
    return ERR_BOUND;	/* more steps than nodes exist: the list is corrupted */

    $ ./bounded_find 35 bounded
    List built. Node 9 now points back to node 4.
    ANOMALY: search exceeded 64 steps. List is corrupted.

The bound is <code>MAX_NODES</code> because a correct list can never have more nodes than exist. If we take more steps than that, something is wrong, and the program says so instead of hanging.

### Rule 3 - No Dynamic Memory After Initialization - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec34/node_pool.c">node_pool.c</a>

In Lecture 32, the telemetry program leaked memory on an early-return path, and in Homework 9 you are writing a deviation to justify your use of <code>malloc</code>. Rule 3 removes the problem at its source. After the program finishes starting up, it does not call <code>malloc</code> or <code>free</code> at all. The reasons:
<ul>
    <li><b>Leaks and use-after-free</b> become impossible if nothing is freed</li>
    <li><b>Running out of memory</b> cannot happen in the middle of a mission if all memory was reserved at startup</li>
    <li><b>Fragmentation:</b> after many allocations and frees, the heap can have plenty of free memory in total but no single block large enough</li>
    <li><b>Timing:</b> how long <code>malloc</code> takes depends on the state of the heap, which breaks the "late answer is a wrong answer" rule</li>
</ul>

So how do we build a linked list? We reserve every node we will ever need in a fixed array, and keep the unused nodes on their own linked list, called the <b>free list</b>:

    static node_t pool[POOL_SIZE];
    static node_t *free_list = NULL;

    /* Replaces malloc: take the first node off the free list, or NULL if empty */
    node_t *pool_alloc(void)
    {
    	if (free_list == NULL) {
    		return NULL;
    	}
    	node_t *node = free_list;
    	free_list = free_list->next_node;
    	node->next_node = NULL;
    	return node;
    }

    /* Replaces free: put the node back at the front of the free list */
    void pool_release(node_t *node)
    {
    	if (node != NULL) {
    		node->next_node = free_list;
    		free_list = node;
    	}
    }

<code>pool_init</code> links all of the nodes onto the free list once, at startup. After that, <code>pool_alloc</code> and <code>pool_release</code> each take the same few steps every time. The example inserts 10 values into a pool of 8:

    $ ./node_pool
    Inserted 0 at address 0x56527076a060
    Inserted 1 at address 0x56527076a070
    ...
    Inserted 7 at address 0x56527076a0d0
    Insert 8 rejected: pool of 8 nodes is full
    Insert 9 rejected: pool of 8 nodes is full
    Removed 7 from address 0x56527076a0d0
    Next allocation returned address 0x56527076a0d0

Look at the addresses. Every node is exactly 16 bytes from the last, because they are all in one array. When the pool is full, the program finds out <i>immediately</i>, in a way you can test before launch, and decides what to do. When a node is released, its memory is reused right away.

The cost is a fixed capacity. You must decide in advance how many nodes the mission could ever need, and justify that number. Flight software teams consider that a feature: the question "how much memory does this need?" gets answered by an engineer during design, not discovered by a crash during flight.

### <font color = "red">Class Introduction Question #3 - Rule 1 forbids recursion. Why does recursion make it impossible for a tool to prove the stack is large enough, and why did <code>recursive_delete</code> fail at 5,000 nodes with a 64 KB stack but not with 8 MB?</font>

### <font color = "red">Class Introduction Question #4 - <code>find_unbounded</code> works correctly whenever the key is in the list. Describe the input that makes it fail, and explain how the bounded version turns a hang into a reported anomaly.</font>

### <font color = "red">Class Introduction Question #5 - Give two reasons flight software avoids <code>malloc</code> after initialization. What happens in <code>node_pool.c</code> when the pool runs out, and why is that better than <code>malloc</code> returning <code>NULL</code> in the middle of a mission?</font>

### In class, we will make a spacecraft command queue flight-ready by applying Rules 1 through 3.